# Chapter 4: Word Embeddings

Before a transformer can process text, each token must become a number, or rather a vector of numbers.
This chapter covers three families of word embeddings:

| Family | Examples | Key property |
|--------|----------|--------------|
| Static (count-based) | GloVe | One fixed vector per word type |
| Static (predictive) | Word2Vec | One fixed vector per word type |
| Contextual | BERT | Vector changes with surrounding context |

All three are implemented as reusable modules in `src/transformer_book_lab/`.
The classic demo is the word **bank**: does it mean a river bank or a financial bank?
Static embeddings cannot tell; BERT can.

In [1]:
import torch
import torch.nn.functional as F

from transformer_book_lab.word2vec_embedder import Word2VecEmbedder
from transformer_book_lab.glove_embedder import GloveEmbedder
from transformer_book_lab.bert_embedder import BertEmbedder

In [2]:
# Shared corpus: Wikipedia-style paragraphs loaded inline
CORPUS = [
    "A bank can refer to a financial institution that accepts deposits and provides loans.",
    "The river bank was covered in mud after the heavy rainfall last weekend.",
    "Neural networks learn distributed representations of words from large text corpora.",
    "Transformers use attention mechanisms to model relationships between tokens in a sequence.",
    "Word embeddings capture semantic similarity so that synonyms end up close in vector space.",
    "The capital of France is Paris, which is also a major center of art and culture.",
    "Deep learning models require large amounts of labeled data for supervised training.",
    "Natural language processing tasks include translation, summarization, and question answering.",
    "The stock market reached a new high after the central bank lowered interest rates.",
    "Researchers fished for trout along the steep bank of the mountain stream.",
    "Recurrent neural networks process sequences token by token, maintaining a hidden state.",
    "Pre-trained language models transfer knowledge from large corpora to downstream tasks.",
]

PROBE_WORD = "bank"
CONTEXT_A  = "I went to the river bank to fish"
CONTEXT_B  = "I went to the bank to deposit money"

## 1. Word2Vec (Skip-Gram)

Word2Vec trains a shallow neural network to predict a target word from its surrounding context
(skip-gram) or to predict context words from a target (CBOW).
After training, the weight matrix of the projection layer becomes the embedding lookup table.

**Key ideas:**

- Words that appear in similar contexts end up with similar vectors.
- Training is unsupervised: the only input is raw text.
- The resulting vectors support arithmetic: `king - man + woman ≈ queen`.

`Word2VecEmbedder` wraps gensim's `Word2Vec` with a PyTorch-friendly interface:
`embed(word)` returns a `torch.Tensor` of shape `(vector_size,)`.

In [3]:
w2v = Word2VecEmbedder(corpus=CORPUS, vector_size=32, window=4, min_count=1)
print(f"Vocabulary size: {w2v.vocab_size}")
print(f"Embedding shape for '{PROBE_WORD}': {w2v.embed(PROBE_WORD).shape}")
print(f"dtype: {w2v.embed(PROBE_WORD).dtype}")

Vocabulary size: 119
Embedding shape for 'bank': torch.Size([32])
dtype: torch.float32


In [4]:
def nearest_neighbours_w2v(embedder: Word2VecEmbedder, word: str, n: int = 5) -> list:
    query = embedder.embed(word)                           # (d,)
    words = list(embedder._model.wv.key_to_index.keys())
    vecs  = torch.tensor(embedder._model.wv.vectors)      # (V, d)
    sims  = F.cosine_similarity(query.unsqueeze(0), vecs, dim=1)
    topk  = sims.topk(n + 1)                              # +1 to skip the query word itself
    results = []
    for idx, score in zip(topk.indices.tolist(), topk.values.tolist()):
        if words[idx] != word:
            results.append((words[idx], round(score, 4)))
    return results[:n]

print(f"Nearest neighbours of '{PROBE_WORD}' (Word2Vec):")
for word, sim in nearest_neighbours_w2v(w2v, PROBE_WORD):
    print(f"  {word:<20} {sim:.4f}")

Nearest neighbours of 'bank' (Word2Vec):
  heavy                0.3170
  weekend.             0.3085
  the                  0.2956
  high                 0.2547
  distributed          0.2402


## 2. GloVe (Global Vectors for Word Representation)

GloVe factorises a global word-co-occurrence matrix rather than training on local windows.
The objective is to predict the log ratio of co-occurrence probabilities,
which captures both global statistics and local context.

**Why GloVe differs from Word2Vec:**

- Word2Vec optimises prediction locally (one context window at a time).
- GloVe optimises a global objective over the full corpus co-occurrence matrix.
- In practice both produce similarly useful embeddings; GloVe tends to be slightly
  more efficient to train on large corpora.

`GloveEmbedder` downloads pre-trained vectors via `gensim.downloader`.
The `glove-wiki-gigaword-50` model was trained on Wikipedia and Gigaword (6B tokens).

> **Note:** The first run downloads ~65 MB. Subsequent runs use a local cache.

In [5]:
glove = GloveEmbedder("glove-wiki-gigaword-50")
print(f"Vector size: {glove.vector_size}")
print(f"Embedding shape for '{PROBE_WORD}': {glove.embed(PROBE_WORD).shape}")
print(f"dtype: {glove.embed(PROBE_WORD).dtype}")

[--------------------------------------------------] 0.0% 0.0/66.0MB downloaded

[--------------------------------------------------] 0.0% 0.0/66.0MB downloaded

[--------------------------------------------------] 0.0% 0.0/66.0MB downloaded

[--------------------------------------------------] 0.0% 0.0/66.0MB downloaded

[--------------------------------------------------] 0.0% 0.0/66.0MB downloaded

...

[=================================================-] 100.0% 66.0/66.0MB downloaded

[=================================================-] 100.0% 66.0/66.0MB downloaded

[=================================================-] 100.0% 66.0/66.0MB downloaded

[==================================================] 100.0% 66.0/66.0MB downloaded

Vector size: 50
Embedding shape for 'bank': torch.Size([50])
dtype: torch.float32


In [6]:
def nearest_neighbours_kv(embedder: GloveEmbedder, word: str, n: int = 5) -> list:
    return [(w, round(float(s), 4)) for w, s in embedder._kv.most_similar(word, topn=n)]

print(f"Nearest neighbours of '{PROBE_WORD}' (GloVe):")
for word, sim in nearest_neighbours_kv(glove, PROBE_WORD):
    print(f"  {word:<20} {sim:.4f}")

Nearest neighbours of 'bank' (GloVe):
  banks                0.8699
  securities           0.7997
  banking              0.7965
  investment           0.7850
  exchange             0.7809


### Comparing nearest neighbours: Word2Vec vs GloVe

Both embedders return a single fixed vector for **bank**, regardless of meaning.
The neighbours reflect the mixture of all senses present in the training corpus.
Word2Vec was trained on the small inline corpus above (12 sentences),
so its vocabulary and quality are limited.
GloVe was trained on billions of tokens, so its neighbours are more informative.

Neither model can separate *river bank* from *financial bank*, because static embeddings
collapse all contexts into one point in vector space.

## 3. BERT Embeddings (Contextual)

BERT (Bidirectional Encoder Representations from Transformers) reads the entire input
sequence at once and produces a different vector for each token depending on its context.

**Architecture sketch:**

```
Input tokens: [CLS]  I  went  to  the  bank  [SEP]
                |    |   |    |    |    |      |
              12 Transformer encoder layers
                |    |   |    |    |    |      |
Last layer:    h0   h1  h2   h3   h4   h5    h6
```

`BertEmbedder.embed(text)` returns the last-layer hidden states for all content tokens
(stripping `[CLS]` and `[SEP]`), shape `(seq_len, 768)` for `bert-base-uncased`.

> **Note:** The first run downloads ~440 MB. Subsequent runs use a local HuggingFace cache.

In [7]:
bert = BertEmbedder("bert-base-uncased")
print(f"d_model: {bert.d_model}")

sample = bert.embed(CONTEXT_A)
print(f'Embedding shape for "{CONTEXT_A}": {sample.shape}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


d_model: 768


Embedding shape for "I went to the river bank to fish": torch.Size([8, 768])


In [8]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def token_index(sentence: str, target: str, tok) -> int:
    tokens = tok.tokenize(sentence)
    return tokens.index(target)

vec_a = bert.embed(CONTEXT_A)   # (seq_len_a, 768)
vec_b = bert.embed(CONTEXT_B)   # (seq_len_b, 768)

idx_a = token_index(CONTEXT_A, PROBE_WORD, tokenizer)
idx_b = token_index(CONTEXT_B, PROBE_WORD, tokenizer)

bank_river   = vec_a[idx_a]   # (768,)
bank_finance = vec_b[idx_b]   # (768,)

similarity = F.cosine_similarity(bank_river.unsqueeze(0), bank_finance.unsqueeze(0)).item()

print(f"Context A: {CONTEXT_A!r}")
print(f"Context B: {CONTEXT_B!r}")
print(f"Cosine similarity of 'bank' across contexts: {similarity:.4f}")
print()
print("A value close to 1.0 would mean the representations are identical (static behaviour).")
print("A value noticeably below 1.0 confirms BERT is encoding different senses.")

Context A: 'I went to the river bank to fish'
Context B: 'I went to the bank to deposit money'
Cosine similarity of 'bank' across contexts: 0.5880

A value close to 1.0 would mean the representations are identical (static behaviour).
A value noticeably below 1.0 confirms BERT is encoding different senses.


## 4. Static vs Contextual Embeddings

The core difference between the two families:

| Property | Word2Vec / GloVe | BERT |
|----------|-----------------|------|
| One vector per word type? | Yes | No |
| Depends on context? | No | Yes |
| Vocabulary fixed after training? | Yes | Yes (subword) |
| Handles OOV words? | No (KeyError) | Yes (subword fallback) |
| Vector shape | `(d,)` | `(seq_len, d)` |
| Typical d | 50 to 300 | 768 to 4096 |
| Training data needed | Millions of tokens | Billions of tokens |
| Inference cost | O(1) lookup | O(seq_len²) attention |

In [9]:
# Static: the same word always returns the same vector
v1 = glove.embed(PROBE_WORD)
v2 = glove.embed(PROBE_WORD)
print("GloVe: same word, same context?")
print(f"  torch.allclose: {torch.allclose(v1, v2)}")   # always True
print()

# Contextual: the same word in different contexts yields different vectors
print("BERT: same word, different context?")
sim_river_finance = F.cosine_similarity(
    bank_river.unsqueeze(0), bank_finance.unsqueeze(0)
).item()
print(f"  cosine similarity (river vs finance): {sim_river_finance:.4f}")
print(f"  vectors are identical: {torch.allclose(bank_river, bank_finance)}")

GloVe: same word, same context?
  torch.allclose: True

BERT: same word, different context?
  cosine similarity (river vs finance): 0.5880
  vectors are identical: False


## Key Takeaways

- **Static embeddings** (Word2Vec, GloVe) assign one fixed vector per word type.
  Training is fast and the resulting vectors are small, but a polysemous word like *bank*
  collapses all its meanings into a single point.

- **Contextual embeddings** (BERT) produce a different vector for every token depending
  on the full input sequence. This allows the model to distinguish *river bank* from
  *financial bank* without any explicit disambiguation signal.

- The shift from static to contextual representations is one of the central advances
  that made large language models possible: pre-training on raw text gives the model
  a rich contextual prior that can be fine-tuned cheaply on downstream tasks.

- All three embedders in this chapter share the same principle:
  map discrete tokens into a continuous vector space where geometry encodes meaning.

## Final Exercise

GloVe vectors support word arithmetic. The classic example is:

```
king - man + woman ≈ queen
```

Use the `GloveEmbedder` loaded above (50-dimensional vectors) to verify this analogy:

1. Retrieve the vectors for *king*, *man*, and *woman*.
2. Compute the analogy vector: `v = embed("king") - embed("man") + embed("woman")`.
3. Find the five words in the GloVe vocabulary whose vectors are closest to `v`
   (exclude *king*, *man*, and *woman* from the results).
4. Does *queen* appear in the top 5? What position?

Implement the search using `F.cosine_similarity` and the full GloVe weight matrix.
**Hint:** `glove._kv.vectors` gives you the full `(V, 50)` numpy array,
and `glove._kv.index_to_key` gives you the corresponding word list.

In [10]:
# Your solution here
